In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv('movies_metadata.csv')
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df=df.drop_duplicates().reset_index(drop=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df=df[['id','title','genres','overview','tagline','vote_average','popularity']]

In [ ]:
df.isnull().sum()

In [ ]:
df=df.dropna(subset=['title']).reset_index(drop=True)

In [ ]:
df['overview']=df['overview'].fillna(' ')

In [ ]:
df.iloc[0]['genres']
import ast

In [ ]:
df['genres']=df['genres'].apply(lambda x:" ".join([i['name'] for i in ast.literal_eval(x)]))

In [ ]:
df.head()

In [ ]:
df['tagline']=df['tagline'].fillna(' ')

In [ ]:
df.isnull().sum()

# DATA GATHERING AND CLEANING
combining multiple columns so that we do not need to convert them into vectors separately. Used for efficiency

In [ ]:
df['tags']=df['overview']+" "+df['genres']+" "+df['tagline']

In [ ]:
df.head()

## NLP

In [ ]:
df.tags[1]

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re

In [ ]:

nltk.download('wordnet')
nltk.download('stopwords')

In [ ]:
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()

In [ ]:
def preprocess(text):
    text=str(text).lower()
    text=re.sub('[^a-zA-Z0-9]',' ',text)
    text=text.split()
    text=[lemmatizer.lemmatize(word) for word in text if not word in stop_words]
    text=" ".join(text)
    return text

In [ ]:
df['tags']=df['tags'].apply(preprocess)

In [ ]:
df.tags[0]

# TEXT VECTORIZATION

In [ ]:
df=df.reset_index(drop=True)

In [ ]:
indices=pd.Series(df.index, index=df['title'])
indices

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
tfidf=TfidfVectorizer(max_features=50000, stop_words='english', ngram_range=(1,2))

In [ ]:
tfidf_matrix=tfidf.fit_transform(df['tags'])

In [ ]:
tfidf_matrix

In [ ]:
def recommend(title,n=10):
    if title not in indices:
        return "Movie not found."
    idx=indices[title]
    sim_scores=cosine_similarity(tfidf_matrix[idx], tfidf_matrix).flatten()
    similar_idx= sim_scores.argsort()[::-1] [1:n+1]
    return df['title'].iloc[similar_idx]

In [ ]:
recommend('Toy Story')

In [ ]:
import pickle

pickle.dump(df, open('movies.pkl','wb'))
pickle.dump(indices, open('indices.pkl','wb'))  
pickle.dump(tfidf_matrix, open('tfidf_matrix.pkl','wb'))
pickle.dump(tfidf, open('tfidf.pkl','wb'))